# Conversion for Mobile Deployment


- PyTorch → ONNX (in Prototype 10)
- ONNX → TensorFlow
- TensorFlow → TFLite + INT8 Quantization

References
- https://github.com/DanieliusKr/onnx-example/blob/main/onnx_example.ipynb
- https://www.geeksforgeeks.org/machine-learning/convert-pytorch-model-to-tf-lite-with-onnx-tf/

### Import

In [3]:
try:
    import google.colab
    IN_COLAB = True
    !pip install onnx2tf
except ImportError:
    IN_COLAB = False

In [5]:
import torch as torch
import torch.nn as nn
import os
import random
import numpy as np

import onnx2tf
import tensorflow as tf

ModuleNotFoundError: No module named 'onnx2tf'

## Conversion to TensorFlow

In [3]:
onnx_path = "convlstm.onnx"
tf_dir = "tf_model"

onnx2tf.convert(
    input_onnx_file_path=onnx_path,
    output_folder_path=tf_dir,
    copy_onnx_input_output_names_to_tflite=True,
    non_verbose=True,
)

# Convert from ONNX to Tensorflow
print(f"Success: conversion to Tensorflow. File is saved at: {tf_dir}")

if IN_COLAB:
  !zip -r 'tf_model.zip' '/content/tf_model'

Success: conversion to Tensorflow. File is saved at: tf_model
  adding: content/tf_model/ (stored 0%)
  adding: content/tf_model/convlstm_float32.tflite (deflated 86%)
  adding: content/tf_model/convlstm_float16.tflite (deflated 86%)
  adding: content/tf_model/fingerprint.pb (stored 0%)
  adding: content/tf_model/variables/ (stored 0%)
  adding: content/tf_model/variables/variables.data-00000-of-00001 (deflated 82%)
  adding: content/tf_model/variables/variables.index (deflated 33%)
  adding: content/tf_model/saved_model.pb (deflated 51%)
  adding: content/tf_model/assets/ (stored 0%)


## Conversion to TFLite

In [4]:
""" Convert from Tensorflow to TFLite """
converter = tf.lite.TFLiteConverter.from_saved_model(tf_dir)
# Dynamic Range Quantization: INT 8 Quantization
converter.optimizations = [tf.lite.Optimize.DEFAULT]
tflite_model = converter.convert()

# Save the TFLite model
with open ("convlstm.tflite", "wb") as f:
    f.write(tflite_model)

print(f"Success: conversion to TFLite. File is saved at: {os.path.join(tf_dir, "convlstm.tflite")}")

Success: conversion to TFLite. File is saved at: tf_model/convlstm.tflite


In [ ]:
def get_input() -> torch.Tensor:
    """ Generate random input data with the same shape as the model's expected input """
    HEIGHT = 128
    WIDTH = 128

    # Generate random video
    # Timesteps, Channels, Height, Width
    video_tensor = torch.rand(20, 3, HEIGHT, WIDTH).to(self.device)
    # Generate random intent
    intent_direction = random.randint(0, 2)
    # Generate random intent position
    intent_position = random.randint(0, 9)

    # Convert random video to frames
    frames = []
    for i in range(video_tensor.shape[0]):
        frame_tensor = video_tensor[i]

        intent_torch = torch.zeros((3, HEIGHT, WIDTH))
        # If intent exists, add intent in its intent position for 1 second (10 frames)
        if intent_position <= i and (intent_position + 10) > i:
            # Convert intent to one-hot vector by filling the specified channel with 1
            intent_torch[intent_direction, :, :] = 1

        intent_torch = intent_torch.to(frame_tensor.device)
        # Append the intent as a channel to the video frame
        frame_tensor = torch.cat((frame_tensor, intent_torch), dim=0)
        frames.append(frame_tensor)

    video_tensor = torch.stack(frames, dim=0)
    # Adds batch dimension to the tensor -> Batch, Timesteps, Channels, Height, Width
    final_video_tensor = video_tensor.unsqueeze(0)
    return final_video_tensor

In [2]:
interpreter = tf.lite.Interpreter(model_content=tflite_model)
interpreter.allocate_tensors()

output_details = interpreter.get_output_details()[0] # Model has single output.
input_details = interpreter.get_input_details()[0] # Model has single input.

input_shape = input_details['shape']
input_data = get_input().cpu().numpy().astype(np.float32) # tf.constant(1., shape=[1, 1])
interpreter.set_tensor(input_details['index'], input_data)
interpreter.invoke()

output = interpreter.get_tensor(output_details['index'])
output_shape = output.shape

print("Interpreted TFLite model")
print(f"Input shape: {input_shape}")
print(f"Output shape: {output_shape}")
print(f"Output: {output}")

NameError: name 'tf' is not defined